In [1]:
print("Hello from SageMaker!")

Hello from SageMaker!


In [2]:
import pandas as pd
import boto3
import sklearn

print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Boto3:", boto3.__version__)

Pandas: 2.3.3
Scikit-learn: 1.7.2
Boto3: 1.43.56


In [5]:
import boto3

s3 = boto3.client("s3")

response = s3.list_buckets()

print("Your S3 buckets:")

for bucket in response["Buckets"]:
    print(bucket["Name"])

Your S3 buckets:
mlops-day2-fatim-2026
sagemaker-eu-north-1-834176180969


In [6]:
import boto3

bucket_name = "mlops-day2-fatim-2026"

response = s3.list_objects_v2(Bucket=bucket_name)

print("Files in bucket:")

for obj in response.get("Contents", []):
    print(obj["Key"])

Files in bucket:
customer_data.csv


In [10]:
import boto3

sts = boto3.client("sts")

identity = sts.get_caller_identity()

print(identity["Arn"])

arn:aws:sts::834176180969:assumed-role/AmazonSageMaker-ExecutionRole-20260918T151840/SageMaker


In [12]:
import boto3

s3 = boto3.client("s3")

bucket = "mlops-day2-fatim-2026"
key = "customer_data.csv"

try:
    response = s3.get_object(Bucket=bucket, Key=key)
    print("SUCCESS! SageMaker can read the S3 file.")
    print(response["Body"].read().decode("utf-8"))
except Exception as e:
    print("ERROR:")
    print(e)

ERROR:
An error occurred (AccessDenied) when calling the GetObject operation: User: arn:aws:sts::834176180969:assumed-role/AmazonSageMaker-ExecutionRole-20260918T151840/SageMaker is not authorized to perform: s3:GetObject on resource: "arn:aws:s3:::mlops-day2-fatim-2026/customer_data.csv" because no identity-based policy allows the s3:GetObject action


In [13]:
import boto3

s3 = boto3.client("s3")

bucket = "mlops-day2-fatim-2026"
key = "customer_data.csv"

try:
    response = s3.get_object(Bucket=bucket, Key=key)
    print("SUCCESS! SageMaker can read the S3 file.")
    print(response["Body"].read().decode("utf-8"))
except Exception as e:
    print("ERROR:")
    print(e)

SUCCESS! SageMaker can read the S3 file.
age,salary,support_calls,churn
25,40000,2,0
35,60000,8,1
29,50000,1,0
42,80000,10,1
31,55000,3,0
45,90000,9,1
28,45000,2,0
38,70000,7,1
30,52000,4,0
50,95000,11,1


In [14]:
import boto3
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")

bucket = "mlops-day2-fatim-2026"
key = "customer_data.csv"

response = s3.get_object(Bucket=bucket, Key=key)

df = pd.read_csv(BytesIO(response["Body"].read()))

print("Data loaded successfully!")
print(df)

Data loaded successfully!
   age  salary  support_calls  churn
0   25   40000              2      0
1   35   60000              8      1
2   29   50000              1      0
3   42   80000             10      1
4   31   55000              3      0
5   45   90000              9      1
6   28   45000              2      0
7   38   70000              7      1
8   30   52000              4      0
9   50   95000             11      1


In [15]:
X = df[["age", "salary", "support_calls"]]
y = df["churn"]

print("Features (X):")
print(X)

print("\nTarget (y):")
print(y)

Features (X):
   age  salary  support_calls
0   25   40000              2
1   35   60000              8
2   29   50000              1
3   42   80000             10
4   31   55000              3
5   45   90000              9
6   28   45000              2
7   38   70000              7
8   30   52000              4
9   50   95000             11

Target (y):
0    0
1    1
2    0
3    1
4    0
5    1
6    0
7    1
8    0
9    1
Name: churn, dtype: int64


In [17]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

print("Model training completed!")

Model training completed!


In [18]:
predictions = model.predict(X)

print("Predictions:")
print(predictions)

Predictions:
[0 1 0 1 0 1 0 1 0 1]


In [19]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y, predictions)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 8
Testing rows: 2


In [21]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [22]:
predictions = model.predict(X_test)

print("Predictions:")
print(predictions)

print("\nActual values:")
print(y_test.values)

Predictions:
[0 1]

Actual values:
[0 1]


In [23]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print("Test Accuracy:", accuracy)

Test Accuracy: 1.0


In [24]:
import joblib

model_path = "model.joblib"

joblib.dump(model, model_path)

print("Model saved successfully!")
print("Saved as:", model_path)

Model saved successfully!
Saved as: model.joblib


In [27]:
import boto3

s3 = boto3.client("s3")

bucket = "mlops-day2-fatim-2026"

s3.upload_file(
    "model.joblib",
    bucket,
    "models/model.joblib"
)

print("Model uploaded to S3 successfully!")

Model uploaded to S3 successfully!
